# Coach DNA Profile Checks

This notebook validates the first-pass Coach DNA scoring outputs and pulls out the first round of interpretable findings.

## Goals
- confirm the exported scoring tables load correctly
- run basic QA on team counts, situation counts, and score ranges
- identify the strongest overall team signals
- inspect which situations drive the strongest differences
- review one team in detail against the league baseline
- generate findings that can later be used in the README or GitHub writeup

In [29]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# Robust project root detection
candidate_paths = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None

for path in candidate_paths:
    if (path / "python").exists() and (path / "outputs").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root from notebook.")

OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROFILE_SEASON = 2025

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_TABLES_DIR:", OUTPUT_TABLES_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)

PROJECT_ROOT: /Users/Tip/Desktop/ea-coach-dna-calibration
OUTPUT_TABLES_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/outputs/tables
PROCESSED_DATA_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/data/processed


In [30]:
ranked_team_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_ranked_team_summary_{PROFILE_SEASON}.csv"
)

ranked_situation_scores = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_ranked_situation_scores_{PROFILE_SEASON}.csv"
)

top_signal_situations = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_top_signal_situations_by_team_{PROFILE_SEASON}.csv"
)

presentation_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_team_summary_presentation_{PROFILE_SEASON}.csv"
)

team_baseline_features = pd.read_csv(
    PROCESSED_DATA_DIR / "team_baseline_features_2025_vs_2023_2025.csv"
)

print("ranked_team_summary:", ranked_team_summary.shape)
print("ranked_situation_scores:", ranked_situation_scores.shape)
print("top_signal_situations:", top_signal_situations.shape)
print("presentation_summary:", presentation_summary.shape)
print("team_baseline_features:", team_baseline_features.shape)

ranked_team_summary: (32, 14)
ranked_situation_scores: (671, 24)
top_signal_situations: (96, 20)
presentation_summary: (32, 14)
team_baseline_features: (671, 85)


## 1. Basic QA Checks
These checks confirm the exported tables look structurally right before interpreting the results.

In [31]:
qa_summary = {
    "team_summary_rows": len(ranked_team_summary),
    "unique_teams_in_team_summary": ranked_team_summary["team"].nunique(),
    "situation_score_rows": len(ranked_situation_scores),
    "unique_teams_in_situation_scores": ranked_situation_scores["team"].nunique(),
    "unique_situations_in_situation_scores": ranked_situation_scores["situation_name"].nunique(),
    "top_signal_rows": len(top_signal_situations),
    "unique_teams_in_top_signals": top_signal_situations["team"].nunique(),
}

pd.Series(qa_summary)

team_summary_rows                         32
unique_teams_in_team_summary              32
situation_score_rows                     671
unique_teams_in_situation_scores          32
unique_situations_in_situation_scores     21
top_signal_rows                           96
unique_teams_in_top_signals               32
dtype: int64

In [32]:
ranked_situation_scores.groupby("team").size().sort_values().head(10)

team
NYJ    20
ARI    21
TB     21
SF     21
SEA    21
PIT    21
PHI    21
NYG    21
NO     21
NE     21
dtype: int64

In [33]:
ranked_situation_scores.groupby("team").size().sort_values(ascending=False).head(10)

team
ARI    21
ATL    21
TEN    21
TB     21
SF     21
SEA    21
PIT    21
PHI    21
NYG    21
NO     21
dtype: int64

## 2. Team-Level Results
Start with the overall rankings to see which teams stand out most strongly in the first-pass scoring model.

In [34]:
ranked_team_summary.head(10)

,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_situation,top_signal_score,lowest_signal_situation,lowest_signal_score
0,1,LA,71.4506,strong_signal,13,62.3970,88.1404,70.5575,71.6600,96.3510,neutral_early_down,83.9844,two_minute_game,34.2305
1,2,BUF,70.8384,strong_signal,13,72.2995,77.7592,70.5647,48.6145,96.0830,third_down,79.5703,fourth_down,45.8760
2,3,WAS,68.2717,strong_signal,13,80.9874,56.2046,64.8656,45.0901,96.2908,tied_early_down,82.3633,two_minute_game,37.3359
3,4,CIN,65.4883,strong_signal,13,74.2507,65.0623,41.7981,56.2964,97.3112,red_zone,82.0703,fourth_down,51.8379
4,5,BAL,61.8666,solid_signal,13,70.4730,54.6368,68.3142,27.7396,95.3569,leading_early_down,70.8594,two_minute_game,22.9512
5,6,CHI,61.8290,solid_signal,13,53.5188,66.4914,65.3718,71.8608,97.1988,third_down,78.3203,fourth_down,35.2266
6,7,NE,59.1905,solid_signal,13,48.3904,77.7351,70.7027,36.2316,96.4879,two_minute_half,79.9609,short_yardage,41.5234
7,8,SEA,59.0944,solid_signal,13,58.3097,59.7254,63.4167,45.3474,96.3373,trailing_one_score,76.9531,two_minute_game,15.0264
8,9,GB,57.8559,solid_signal,13,47.3359,67.0709,70.3521,55.6716,96.1749,third_down,83.2422,tied_early_down,25.4297
9,10,SF,57.1461,solid_signal,13,48.3846,72.8378,45.9918,63.2382,96.6462,third_down,66.8945,fourth_down,33.5566


### How to read this table

This table shows the teams with the strongest overall Coach DNA signal in the first-pass model.

#### What the key numbers mean

- `overall_coach_dna_score`  
  Higher means the team shows a stronger and more credible situational identity relative to the league baseline.  
  For CPU tuning, a higher score suggests the team has more evidence supporting a distinct coaching profile.

- `tendency_signal_score_avg`  
  Higher means the team behaves more differently from baseline.  
  This is important for tuning because it points to where the CPU should feel less generic.

- `efficiency_signal_score_avg`  
  Higher means the team’s distinct behavior tends to work better than baseline.  
  This is important because it helps determine which differences are worth preserving.

- `top_signal_situation`  
  This shows the situation where the team’s coaching DNA appears most clearly.

#### Why this matters for CPU-controlled coaching decisions

A high-ranking team in this table is a team where the CPU should probably not be tuned as a league-average generalist.  
Instead, it suggests the team has a stronger case for:
- situation-specific run/pass behavior
- distinct tempo patterns
- unique structural tendencies
- differentiated situational aggression

### Practical takeaway

This table shows which teams have the strongest evidence for **custom CPU coach tuning** rather than generic default behavior.

In [35]:
ranked_team_summary.tail(10)

,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_situation,top_signal_score,lowest_signal_situation,lowest_signal_score
22,23,CAR,46.4753,moderate_signal,13,52.4559,35.9621,38.8303,49.9453,93.9939,leading_one_score,59.9219,two_minute_game,31.3154
23,24,NO,45.5841,moderate_signal,13,56.9997,28.9181,36.3539,43.0440,92.9593,two_minute_game,68.4281,fourth_down,21.2959
24,25,MIN,44.1784,developing_signal,13,47.7545,36.4424,53.4584,20.2193,95.1590,red_zone,59.6875,fourth_down,28.1660
25,26,HOU,44.1350,developing_signal,13,43.8258,30.9634,41.4845,69.7000,96.0942,two_minute_half,59.0625,fourth_down,24.7246
26,27,PIT,43.8918,developing_signal,13,33.9831,51.0297,38.7492,61.9160,96.6330,leading_early_down,58.6133,trailing_early_down,30.1953
27,28,LAC,43.2726,developing_signal,13,44.0018,40.3848,38.7825,36.0798,96.6396,third_down,56.4453,fourth_down,31.6084
28,29,NYJ,40.0831,developing_signal,13,53.3315,21.3194,33.0184,32.5823,91.7325,short_yardage,51.0258,red_zone,17.5000
29,30,TEN,37.1841,developing_signal,13,39.6232,24.5221,31.1616,49.5500,94.8670,tied_early_down,51.0156,trailing_one_score,26.4844
30,31,CLE,33.9912,developing_signal,13,37.3563,17.5156,28.6021,45.1110,96.0123,short_yardage,49.1273,goal_to_go,20.8266
31,32,LV,33.6152,developing_signal,13,43.3354,17.0911,29.4971,28.7341,90.6494,trailing_one_score,52.6172,fourth_down,20.0508


In [36]:
ranked_team_summary["score_tier"].value_counts()

score_tier
solid_signal         10
moderate_signal      10
developing_signal     8
strong_signal         4
Name: count, dtype: int64

In [37]:
ranked_team_summary[[
    "team",
    "overall_coach_dna_score",
    "tendency_signal_score_avg",
    "efficiency_signal_score_avg",
    "explosiveness_signal_score_avg",
    "stability_signal_score_avg",
    "top_signal_situation",
    "top_signal_score",
]].head(15)

,team,overall_coach_dna_score,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,top_signal_situation,top_signal_score
0,LA,71.4506,62.3970,88.1404,70.5575,71.6600,neutral_early_down,83.9844
1,BUF,70.8384,72.2995,77.7592,70.5647,48.6145,third_down,79.5703
2,WAS,68.2717,80.9874,56.2046,64.8656,45.0901,tied_early_down,82.3633
3,CIN,65.4883,74.2507,65.0623,41.7981,56.2964,red_zone,82.0703
4,BAL,61.8666,70.4730,54.6368,68.3142,27.7396,leading_early_down,70.8594
5,CHI,61.8290,53.5188,66.4914,65.3718,71.8608,third_down,78.3203
6,NE,59.1905,48.3904,77.7351,70.7027,36.2316,two_minute_half,79.9609
7,SEA,59.0944,58.3097,59.7254,63.4167,45.3474,trailing_one_score,76.9531
8,GB,57.8559,47.3359,67.0709,70.3521,55.6716,third_down,83.2422
9,SF,57.1461,48.3846,72.8378,45.9918,63.2382,third_down,66.8945


## 3. Situation-Level Patterns
These checks show which situations tend to generate the strongest signals across the league.

In [38]:
situation_strength = (
    ranked_situation_scores
    .groupby("situation_name", as_index=False)
    .agg(
        avg_adjusted_score=("coach_dna_score_adjusted", "mean"),
        max_adjusted_score=("coach_dna_score_adjusted", "max"),
        min_adjusted_score=("coach_dna_score_adjusted", "min"),
        avg_team_play_count=("team_play_count", "mean"),
        teams=("team", "nunique"),
    )
    .sort_values("avg_adjusted_score", ascending=False)
)

situation_strength

,situation_name,avg_adjusted_score,max_adjusted_score,min_adjusted_score,avg_team_play_count,teams
0,all_offense,53.984375,84.296875,22.773438,1025.437500,32
9,one_score,53.984375,82.734375,22.949219,680.468750,32
17,trailing_one_score,53.984375,76.953125,26.484375,282.406250,32
16,trailing_early_down,53.984375,79.648438,28.437500,376.687500,32
15,trailing,53.984375,78.750000,25.468750,495.406250,32
13,tied,53.984375,86.953125,32.656250,193.531250,32
12,third_down,53.984375,83.242188,25.937500,212.531250,32
1,early_down,53.984375,82.031250,25.312500,785.343750,32
10,red_zone,53.984375,82.070312,17.500000,160.656250,32
8,neutral_early_down,53.984375,83.984375,28.867188,427.000000,32


### How to read this table

This table shows which situations are most useful for detecting coaching identity across teams.

#### What the key numbers mean

- `avg_adjusted_score`  
  Higher means that situation tends to separate teams more clearly from one another.

- `avg_team_play_count`  
  Higher means there is enough sample to trust the signal more confidently.

- `max_adjusted_score`  
  Higher means at least one team stands out very clearly in that situation.

#### Why this matters for CPU-controlled coaching decisions

Not every situation is equally valuable for tuning.  
Some situations reveal team identity much more clearly than others.

The strongest tuning situations are the ones where:
- teams behave differently
- the sample is large enough
- the behavior reflects real pressure or decision logic

These are the best candidates for:
- playcall weighting adjustments
- score-state tuning
- down-and-distance behavior changes
- team personality logic

### Practical takeaway

This table helps identify the situations that matter most when building **team-specific CPU coaching profiles**.

In [39]:
situation_variance = (
    ranked_situation_scores
    .groupby("situation_name", as_index=False)
    .agg(
        score_std=("coach_dna_score_adjusted", "std"),
        score_range=("coach_dna_score_adjusted", lambda s: s.max() - s.min()),
        avg_sample=("team_play_count", "mean"),
    )
    .sort_values("score_std", ascending=False)
)

situation_variance

,situation_name,score_std,score_range,avg_sample
0,all_offense,16.730030,61.523438,1025.437500
1,early_down,16.272480,56.718750,785.343750
12,third_down,15.368308,57.304688,212.531250
20,two_minute_half,15.076770,52.500000,120.281250
18,trailing_two_plus_scores,14.943507,55.429688,213.000000
4,leading,14.924269,56.074219,336.500000
9,one_score,14.907412,59.785156,680.468750
6,leading_one_score,14.879474,52.704687,204.531250
5,leading_early_down,14.498576,51.239844,259.437500
10,red_zone,14.461601,64.570312,160.656250


## 4. Top Signal Situations by Team
This helps identify the situations where each team looks most distinct relative to baseline.

In [40]:
top_signal_situations.head(20)

,team,team_rank_within_top_signals,situation_name,team_play_count,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,formation_profile_label,tempo_profile_label,efficiency_profile_label
0,ARI,1,two_minute_half,128,62.402344,57.81250,57.421875,79.166667,51.56250,0.0854,-0.0854,0.0519,0.0085,0.0206,0.0066,0.0448,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,close_to_baseline_tempo,close_to_baseline_efficiency
1,ARI,2,red_zone,165,61.289062,78.12500,38.281250,66.666667,15.62500,0.1880,-0.1880,0.1411,0.0328,-0.0802,-0.0345,0.0215,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,less_efficient_than_baseline
2,ARI,3,trailing_two_plus_scores,355,59.335938,54.68750,60.156250,53.125000,67.18750,0.0582,-0.0582,0.0334,0.0683,0.0755,0.0211,0.0094,more_dropback_heavy_than_baseline,close_to_baseline_shotgun_usage,faster_than_baseline,more_efficient_than_baseline
3,ATL,1,two_minute_half,131,71.914062,66.40625,65.625000,76.041667,92.18750,-0.0860,0.0860,0.0153,-0.0718,0.0466,0.0120,0.0256,more_run_heavy_than_baseline,close_to_baseline_shotgun_usage,slower_than_baseline,more_efficient_than_baseline
4,ATL,2,trailing_one_score,255,69.179688,64.06250,78.906250,55.208333,73.43750,0.0424,-0.0424,0.1739,-0.0107,0.1234,0.0548,0.0113,close_to_baseline_run_pass_split,more_shotgun_than_baseline,close_to_baseline_tempo,more_efficient_than_baseline
5,ATL,3,goal_to_go,60,66.248438,65.62500,81.250000,73.958333,86.71875,-0.0757,0.0757,0.2390,0.0472,0.3654,0.0287,0.0093,more_run_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
6,BAL,1,tied,183,71.621094,78.90625,67.578125,52.083333,64.06250,-0.0986,0.0986,-0.0937,-0.0661,0.0633,0.0485,-0.0050,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
7,BAL,2,early_down,728,71.601562,79.68750,66.406250,80.208333,21.09375,-0.1015,0.1015,-0.0516,-0.0811,0.0372,0.0125,0.0269,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
8,BAL,3,all_offense,954,71.386719,79.68750,68.359375,76.041667,20.31250,-0.0873,0.0873,-0.0468,-0.0745,0.0393,0.0110,0.0202,more_run_heavy_than_baseline,close_to_baseline_shotgun_usage,slower_than_baseline,more_efficient_than_baseline
9,BUF,1,all_offense,1056,84.296875,82.03125,94.531250,91.666667,50.00000,-0.0680,0.0680,-0.1948,-0.0423,0.1464,0.0532,0.0263,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline


### How to read this table

This table shows the situations where each team’s coaching DNA appears most strongly.

#### What the key numbers mean

- `coach_dna_score_adjusted`  
  Higher means this situation is one of the clearest places where the team’s identity shows up.

- `tendency_signal_score`  
  Higher means the team behaves more differently from the baseline.

- `efficiency_signal_score`  
  Higher means the team is outperforming the baseline in that same situation.

- profile labels  
  These describe the form of the difference:
  - more run-heavy
  - more dropback-heavy
  - more shotgun
  - faster tempo
  - more efficient than baseline

#### Why this matters for CPU-controlled coaching decisions

This is one of the most useful outputs in the project for a tuning conversation.  
It points directly to where a team should feel different from a default CPU coach.

Examples:
- a team whose strongest signal is `neutral_early_down` may need a custom early-down identity
- a team whose strongest signal is `red_zone` may need custom goal-line or condensed-field tuning
- a team whose strongest signal is `trailing_one_score` may need more specific comeback behavior

### Practical takeaway

This table shows where each team’s **CPU coaching identity is most worth tuning**.

In [41]:
top_signal_situations["situation_name"].value_counts().head(15)

situation_name
two_minute_half             10
red_zone                     8
third_down                   7
short_yardage                7
trailing_two_plus_scores     6
trailing_one_score           6
tied                         6
all_offense                  6
leading_early_down           6
leading                      6
tied_early_down              6
leading_one_score            4
trailing_early_down          4
early_down                   3
trailing                     3
Name: count, dtype: int64

In [42]:
top_signal_situations["tendency_profile_label"].value_counts()

tendency_profile_label
more_run_heavy_than_baseline         37
close_to_baseline_run_pass_split     31
more_dropback_heavy_than_baseline    28
Name: count, dtype: int64

In [43]:
top_signal_situations["efficiency_profile_label"].value_counts()

efficiency_profile_label
more_efficient_than_baseline    71
less_efficient_than_baseline    15
close_to_baseline_efficiency    10
Name: count, dtype: int64

## 5. Team Deep Dive
Change the `TEAM_CODE` below to inspect any team in more detail.

In [44]:
TEAM_CODE = "BUF"

In [45]:
team_summary_view = ranked_team_summary.loc[ranked_team_summary["team"] == TEAM_CODE]
team_summary_view

,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_situation,top_signal_score,lowest_signal_situation,lowest_signal_score
1,2,BUF,70.8384,strong_signal,13,72.2995,77.7592,70.5647,48.6145,96.083,third_down,79.5703,fourth_down,45.876


In [46]:
team_situations = (
    ranked_situation_scores.loc[ranked_situation_scores["team"] == TEAM_CODE]
    .sort_values("situation_order")
)

team_situations[[
    "team",
    "situation_name",
    "team_play_count",
    "coach_dna_score_adjusted",
    "tendency_signal_score",
    "efficiency_signal_score",
    "explosiveness_signal_score",
    "stability_signal_score",
    "dropback_rate_delta",
    "rush_rate_delta",
    "shotgun_rate_delta",
    "no_huddle_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

,team,situation_name,team_play_count,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,efficiency_profile_label
1,BUF,all_offense,1056,84.296875,82.031250,94.531250,91.666667,50.000000,-0.0680,0.0680,-0.1948,-0.0423,0.1464,0.0532,0.0263,more_run_heavy_than_baseline,more_efficient_than_baseline
7,BUF,early_down,819,82.031250,80.468750,89.843750,84.895833,56.250000,-0.0679,0.0679,-0.2138,-0.0451,0.1182,0.0521,0.0240,more_run_heavy_than_baseline,more_efficient_than_baseline
15,BUF,third_down,205,79.570312,78.125000,94.531250,82.291667,34.375000,-0.0762,0.0762,-0.1197,-0.0290,0.1919,0.0538,0.0303,more_run_heavy_than_baseline,more_efficient_than_baseline
430,BUF,fourth_down,32,45.875977,50.390625,75.781250,69.791667,63.281250,0.0650,-0.0650,-0.0812,-0.0630,0.5230,0.0495,0.0666,more_dropback_heavy_than_baseline,more_efficient_than_baseline
281,BUF,short_yardage,128,55.078125,46.875000,64.062500,60.416667,39.062500,-0.0429,0.0429,-0.2041,0.0233,0.0047,0.0289,0.0032,close_to_baseline_run_pass_split,close_to_baseline_efficiency
34,BUF,red_zone,190,75.312500,77.343750,91.406250,43.229167,61.718750,-0.0871,0.0871,-0.2484,0.0529,0.1341,0.0723,-0.0091,more_run_heavy_than_baseline,more_efficient_than_baseline
80,BUF,goal_to_go,76,70.045313,93.750000,82.812500,39.062500,50.781250,-0.1468,0.1468,-0.3347,0.1376,0.1853,0.1015,-0.0074,more_run_heavy_than_baseline,more_efficient_than_baseline
119,BUF,two_minute_half,91,66.424219,89.062500,74.218750,57.812500,25.000000,-0.1415,0.1415,-0.2404,-0.0535,0.1188,0.0384,0.0158,more_run_heavy_than_baseline,more_efficient_than_baseline
269,BUF,two_minute_game,51,55.807031,64.062500,67.968750,56.250000,37.500000,-0.1328,0.1328,-0.1898,-0.0157,0.1411,0.0568,0.0105,more_run_heavy_than_baseline,more_efficient_than_baseline
18,BUF,one_score,649,78.750000,76.562500,89.062500,94.791667,28.125000,-0.0565,0.0565,-0.1728,-0.0370,0.1083,0.0454,0.0348,more_run_heavy_than_baseline,more_efficient_than_baseline


### How to read this table

This section shows how one team’s behavior differs from the league baseline across all major situations. This is the best section for translating the model into possible CPU tuning ideas.

#### What the key numbers mean

- positive tendency deltas  
  These show where the team behaves differently from baseline in structure, run-pass balance, or tempo.

- positive `avg_epa_delta` and `success_rate_delta`  
  These suggest the team’s tendency is producing stronger results than baseline, which makes it more credible as a tuning signal.

- positive `explosive_play_rate_delta`  
  This suggests the team creates more chunk-play pressure than baseline in that situation.

- `coach_dna_score_adjusted`  
  This summarizes how strong and reliable the overall signal is for that team-situation combination.

#### Why this matters for CPU-controlled coaching decisions

This section can directly inform questions like:
- should this team call more run or pass in this game state?
- should this team lean more heavily into shotgun?
- should this team play faster when trailing?
- should this team feel more conservative or more aggressive than the baseline CPU coach?

### Practical takeaway

This table is where the project becomes most useful for tuning discussions because it shows **how one specific team should behave differently from league-average CPU logic by situation**.

In [47]:
team_situations.sort_values("coach_dna_score_adjusted", ascending=False).head(10)[[
    "situation_name",
    "team_play_count",
    "coach_dna_score_adjusted",
    "dropback_rate_delta",
    "rush_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

,situation_name,team_play_count,coach_dna_score_adjusted,dropback_rate_delta,rush_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,efficiency_profile_label
1,all_offense,1056,84.296875,-0.0680,0.0680,0.1464,0.0532,0.0263,more_run_heavy_than_baseline,more_efficient_than_baseline
7,early_down,819,82.031250,-0.0679,0.0679,0.1182,0.0521,0.0240,more_run_heavy_than_baseline,more_efficient_than_baseline
8,trailing_two_plus_scores,234,81.914062,-0.0958,0.0958,0.2178,0.0676,0.0252,more_run_heavy_than_baseline,more_efficient_than_baseline
13,tied,174,79.921875,-0.0982,0.0982,0.1124,0.0682,0.0532,more_run_heavy_than_baseline,more_efficient_than_baseline
15,third_down,205,79.570312,-0.0762,0.0762,0.1919,0.0538,0.0303,more_run_heavy_than_baseline,more_efficient_than_baseline
18,one_score,649,78.750000,-0.0565,0.0565,0.1083,0.0454,0.0348,more_run_heavy_than_baseline,more_efficient_than_baseline
20,leading,384,78.671875,-0.0665,0.0665,0.1289,0.0497,0.0153,more_run_heavy_than_baseline,more_efficient_than_baseline
24,leading_early_down,293,78.007812,-0.0714,0.0714,0.1640,0.0611,0.0272,more_run_heavy_than_baseline,more_efficient_than_baseline
26,tied_early_down,141,77.304688,-0.0799,0.0799,0.0230,0.0476,0.0630,more_run_heavy_than_baseline,close_to_baseline_efficiency
31,trailing,498,76.015625,-0.0546,0.0546,0.1703,0.0516,0.0250,more_run_heavy_than_baseline,more_efficient_than_baseline


In [48]:
team_situations.sort_values("coach_dna_score_adjusted", ascending=True).head(10)[[
    "situation_name",
    "team_play_count",
    "coach_dna_score_adjusted",
    "dropback_rate_delta",
    "rush_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

,situation_name,team_play_count,coach_dna_score_adjusted,dropback_rate_delta,rush_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,efficiency_profile_label
430,fourth_down,32,45.875977,0.0650,-0.0650,0.5230,0.0495,0.0666,more_dropback_heavy_than_baseline,more_efficient_than_baseline
281,short_yardage,128,55.078125,-0.0429,0.0429,0.0047,0.0289,0.0032,close_to_baseline_run_pass_split,close_to_baseline_efficiency
269,two_minute_game,51,55.807031,-0.1328,0.1328,0.1411,0.0568,0.0105,more_run_heavy_than_baseline,more_efficient_than_baseline
198,trailing_one_score,264,60.078125,-0.0248,0.0248,0.1325,0.0380,0.0253,close_to_baseline_run_pass_split,more_efficient_than_baseline
119,two_minute_half,91,66.424219,-0.1415,0.1415,0.1188,0.0384,0.0158,more_run_heavy_than_baseline,more_efficient_than_baseline
80,goal_to_go,76,70.045313,-0.1468,0.1468,0.1853,0.1015,-0.0074,more_run_heavy_than_baseline,more_efficient_than_baseline
78,leading_one_score,211,70.312500,-0.0602,0.0602,0.0738,0.0367,0.0312,more_run_heavy_than_baseline,more_efficient_than_baseline
68,trailing_early_down,385,71.289062,-0.0574,0.0574,0.1175,0.0477,0.0070,more_run_heavy_than_baseline,more_efficient_than_baseline
40,leading_two_plus_scores,173,74.233871,-0.0632,0.0632,0.1950,0.0657,-0.0046,more_run_heavy_than_baseline,more_efficient_than_baseline
34,red_zone,190,75.312500,-0.0871,0.0871,0.1341,0.0723,-0.0091,more_run_heavy_than_baseline,more_efficient_than_baseline


## 6. Sample Size Guardrails
These checks help keep us honest about situations that may be noisy because of smaller samples.

### How to read this section

These tables show where the project’s signals are backed by strong sample and where they are more fragile.

#### Why this matters for CPU-controlled coaching decisions

A team might look very distinctive in a tiny sample, but that does not always mean the signal is strong enough to tune around.

For design purposes:
- large, stable samples are better candidates for tuning
- smaller samples can still be interesting, but they should carry less weight

#### Practical takeaway

The best CPU tuning candidates are situations where:
- the team differs from baseline
- the behavior appears effective
- and the sample is strong enough to trust

In [49]:
ranked_situation_scores["team_sample_quality"].value_counts()

team_sample_quality
strong       538
good          70
thin          61
very_thin      2
Name: count, dtype: int64

In [50]:
ranked_situation_scores.loc[
    ranked_situation_scores["team_sample_quality"].isin(["thin", "very_thin"])
].sort_values(["team_sample_quality", "team_play_count", "team"]).head(30)

,rank,team,situation_order,situation_name,team_play_count,team_sample_quality,coach_dna_score_adjusted,coach_dna_score_raw,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,sample_reliability_score,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,formation_profile_label,tempo_profile_label,efficiency_profile_label
342,343,CIN,4,fourth_down,20,thin,51.837891,69.117188,63.671875,82.421875,54.687500,89.062500,55.0,0.0775,-0.0775,0.1250,-0.0943,0.8741,0.1557,0.0291,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
570,571,DEN,4,fourth_down,20,thin,35.944336,47.925781,44.531250,37.109375,72.916667,49.218750,55.0,-0.0225,0.0225,-0.2750,-0.0443,0.1821,-0.0443,0.0791,close_to_baseline_run_pass_split,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
612,613,LAC,4,fourth_down,20,thin,31.608398,42.144531,22.656250,48.046875,85.416667,43.750000,55.0,-0.0225,0.0225,-0.0250,-0.0443,0.0821,0.0057,0.0791,close_to_baseline_run_pass_split,close_to_baseline_shotgun_usage,slower_than_baseline,more_efficient_than_baseline
382,383,PHI,4,fourth_down,20,thin,49.728516,66.304688,69.921875,60.546875,53.645833,89.062500,55.0,-0.1725,0.1725,-0.1250,-0.0443,0.4490,0.0057,-0.0209,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
595,596,SF,4,fourth_down,22,thin,33.556641,44.742188,45.703125,50.390625,25.000000,50.781250,55.0,-0.0316,0.0316,-0.1750,-0.0488,0.2322,0.0466,-0.0754,close_to_baseline_run_pass_split,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
617,618,MIA,4,fourth_down,23,thin,31.183594,41.578125,55.468750,27.343750,22.395833,36.718750,55.0,0.1166,-0.1166,0.0641,-0.0508,-0.5260,-0.1095,-0.0774,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,slower_than_baseline,less_efficient_than_baseline
492,493,PIT,4,fourth_down,23,thin,41.071289,54.761719,65.625000,33.984375,47.395833,68.750000,55.0,-0.1442,0.1442,-0.1533,0.0361,-0.0753,-0.0226,-0.0339,more_run_heavy_than_baseline,less_shotgun_than_baseline,faster_than_baseline,less_efficient_than_baseline
531,532,TEN,16,leading_two_plus_scores,23,thin,38.761089,51.681452,61.290323,25.403226,89.247312,16.129032,55.0,-0.1520,0.1520,-0.0720,-0.0151,-0.4330,-0.0343,0.0364,more_run_heavy_than_baseline,less_shotgun_than_baseline,close_to_baseline_tempo,less_efficient_than_baseline
636,637,MIN,4,fourth_down,24,thin,28.166016,37.554688,5.859375,71.484375,72.916667,33.593750,55.0,0.0025,-0.0025,-0.0083,-0.0110,0.0710,0.0390,0.0458,close_to_baseline_run_pass_split,close_to_baseline_shotgun_usage,close_to_baseline_tempo,more_efficient_than_baseline
400,401,BAL,4,fourth_down,25,thin,48.000000,64.000000,56.640625,79.296875,70.833333,53.125000,55.0,0.1375,-0.1375,0.0450,-0.0543,0.5297,0.0557,0.0391,more_dropback_heavy_than_baseline,close_to_baseline_shotgun_usage,slower_than_baseline,more_efficient_than_baseline


In [51]:
small_sample_summary = (
    ranked_situation_scores
    .groupby(["situation_name", "team_sample_quality"], as_index=False)
    .size()
    .sort_values(["situation_name", "team_sample_quality"])
)

small_sample_summary.head(50)

,situation_name,team_sample_quality,size
0,all_offense,strong,32
1,early_down,strong,32
2,fourth_down,thin,30
3,fourth_down,very_thin,2
4,goal_to_go,good,28
5,goal_to_go,thin,4
6,leading,good,1
7,leading,strong,31
8,leading_early_down,good,2
9,leading_early_down,strong,29


## 7. Quick Finding Builder
These tables help turn the output into portfolio-ready statements.

In [52]:
top_overall = ranked_team_summary.head(5)[["rank", "team", "overall_coach_dna_score", "top_signal_situation", "top_signal_score"]]
top_overall

,rank,team,overall_coach_dna_score,top_signal_situation,top_signal_score
0,1,LA,71.4506,neutral_early_down,83.9844
1,2,BUF,70.8384,third_down,79.5703
2,3,WAS,68.2717,tied_early_down,82.3633
3,4,CIN,65.4883,red_zone,82.0703
4,5,BAL,61.8666,leading_early_down,70.8594


In [53]:
best_situations = (
    ranked_situation_scores
    .sort_values("coach_dna_score_adjusted", ascending=False)
    .head(15)[[
        "rank",
        "team",
        "situation_name",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]]
)

best_situations

,rank,team,situation_name,team_play_count,coach_dna_score_adjusted,tendency_profile_label,efficiency_profile_label
0,1,WAS,tied,136,86.953125,more_run_heavy_than_baseline,more_efficient_than_baseline
1,2,BUF,all_offense,1056,84.296875,more_run_heavy_than_baseline,more_efficient_than_baseline
2,3,LA,neutral_early_down,448,83.984375,more_dropback_heavy_than_baseline,more_efficient_than_baseline
3,4,GB,third_down,206,83.242188,more_run_heavy_than_baseline,more_efficient_than_baseline
4,5,LA,one_score,739,82.734375,more_dropback_heavy_than_baseline,more_efficient_than_baseline
5,6,WAS,tied_early_down,104,82.363281,more_run_heavy_than_baseline,close_to_baseline_efficiency
6,7,CIN,red_zone,146,82.070312,more_dropback_heavy_than_baseline,more_efficient_than_baseline
7,8,BUF,early_down,819,82.031250,more_run_heavy_than_baseline,more_efficient_than_baseline
8,9,BUF,trailing_two_plus_scores,234,81.914062,more_run_heavy_than_baseline,more_efficient_than_baseline
9,10,LA,short_yardage,129,81.875000,more_dropback_heavy_than_baseline,more_efficient_than_baseline


In [54]:
most_run_heavy = (
    ranked_situation_scores
    .sort_values("rush_rate_delta", ascending=False)
    .head(15)[[
        "team",
        "situation_name",
        "team_play_count",
        "rush_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
    ]]
)

most_run_heavy

,team,situation_name,team_play_count,rush_rate_delta,avg_epa_delta,success_rate_delta
393,JAX,fourth_down,29,0.2777,-0.2789,0.0074
273,IND,fourth_down,27,0.2521,0.3578,0.1224
356,JAX,two_minute_game,36,0.1818,-0.2651,-0.0331
297,DEN,two_minute_game,34,0.1818,0.0153,-0.0118
382,PHI,fourth_down,20,0.1725,0.4490,0.0057
349,HOU,two_minute_game,45,0.1707,0.0737,-0.0164
580,PHI,two_minute_game,48,0.1610,-0.1998,-0.1025
327,ATL,leading_two_plus_scores,74,0.1590,-0.2159,-0.0337
490,NYJ,leading_one_score,63,0.1531,-0.3198,-0.1103
531,TEN,leading_two_plus_scores,23,0.1520,-0.4330,-0.0343


In [55]:
most_dropback_heavy = (
    ranked_situation_scores
    .sort_values("dropback_rate_delta", ascending=False)
    .head(15)[[
        "team",
        "situation_name",
        "team_play_count",
        "dropback_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
    ]]
)

most_dropback_heavy

,team,situation_name,team_play_count,dropback_rate_delta,avg_epa_delta,success_rate_delta
380,ARI,two_minute_game,40,0.2432,-0.1142,-0.0192
254,WAS,fourth_down,26,0.2237,0.3523,0.0711
105,NO,two_minute_game,59,0.2165,0.0987,0.1312
185,ARI,red_zone,165,0.1880,-0.0802,-0.0345
411,PIT,two_minute_game,37,0.1831,-0.1011,-0.0428
473,LV,goal_to_go,40,0.1743,-0.1001,-0.1130
417,DAL,fourth_down,34,0.1716,0.4734,0.1028
188,CIN,goal_to_go,61,0.1636,-0.2580,-0.0282
319,KC,goal_to_go,78,0.1525,-0.2023,-0.0790
438,NYJ,two_minute_game,54,0.1515,-0.2088,-0.0794


## 8. Notes
Use this section to write down 5–10 findings that are worth carrying into the README, GitHub post, or interview narrative.

### Draft findings
- 
- 
- 
- 
- 